In [ ]:
# ------------------------
# Standard Library Imports
# ------------------------
import os
import re
import ast
import json
from functools import reduce

# ------------------------
# Third-Party Imports
# ------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# FastAI
from fastai.learner import Learner
from fastai.data.core import DataLoaders
from fastai.metrics import accuracy
from fastai.losses import CrossEntropyLossFlat
from fastai.callback.all import SaveModelCallback, EarlyStoppingCallback

# MONAI
from monai.networks.nets import resnet18

# XGBoost
import xgboost as xgb

# scikit-learn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

%matplotlib inline

In [ ]:
import pandas as pd

###########################################################
# 1) Load Aortic Distensibility
###########################################################
asc_aorta_dist = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24120.csv')
asc_aorta_dist = asc_aorta_dist.rename(
    columns={'eid': 'IID', '24120-2.0': 'asc_aorta_dist'}
)[['IID', 'asc_aorta_dist']]

desc_aorta_dist = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24123.csv')
desc_aorta_dist = desc_aorta_dist.rename(
    columns={'eid': 'IID', '24123-2.0': 'desc_aorta_dist'}
)[['IID', 'desc_aorta_dist']]

###########################################################
# 2) Load LV Structure & Function
###########################################################
# LV ejection fraction
lv_ef = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24103.csv')
lv_ef = lv_ef.rename(columns={'eid': 'IID', '24103-2.0': 'lv_ejection_fraction'})[['IID', 'lv_ejection_fraction']]

# LV end diastolic volume
lv_edv = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24100.csv')
lv_edv = lv_edv.rename(columns={'eid': 'IID', '24100-2.0': 'lv_end_diastolic_vol'})[['IID', 'lv_end_diastolic_vol']]

# LV end systolic volume
lv_esv = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24101.csv')
lv_esv = lv_esv.rename(columns={'eid': 'IID', '24101-2.0': 'lv_end_systolic_vol'})[['IID', 'lv_end_systolic_vol']]

# LV stroke volume
lv_sv = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24102.csv')
lv_sv = lv_sv.rename(columns={'eid': 'IID', '24102-2.0': 'lv_stroke_vol'})[['IID', 'lv_stroke_vol']]

# LV myocardial mass
lv_mass = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24105.csv')
lv_mass = lv_mass.rename(columns={'eid': 'IID', '24105-2.0': 'lv_myocardial_mass'})[['IID', 'lv_myocardial_mass']]

###########################################################
# 3) Load LV Deformation (Strain) Metrics
###########################################################
# LV circumferential strain (global)
lv_circ_strain_global = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24157.csv')
lv_circ_strain_global = lv_circ_strain_global.rename(
    columns={'eid': 'IID', '24157-2.0': 'lv_circ_strain_global'}
)[['IID', 'lv_circ_strain_global']]

# LV radial strain (global)
lv_rad_strain_global = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24174.csv')
lv_rad_strain_global = lv_rad_strain_global.rename(
    columns={'eid': 'IID', '24174-2.0': 'lv_rad_strain_global'}
)[['IID', 'lv_rad_strain_global']]

# LV longitudinal strain (global)
lv_long_strain_global = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24181.csv')
lv_long_strain_global = lv_long_strain_global.rename(
    columns={'eid': 'IID', '24181-2.0': 'lv_long_strain_global'}
)[['IID', 'lv_long_strain_global']]

###########################################################
# 4) Load LV Wall Thickness (Global)
###########################################################
lv_wall_thick_global = pd.read_csv('/corral/utexas/UKB-Imaging-Genetics/temp_imaging_data/pheno_split_into_files_011924/fid24140.csv')
lv_wall_thick_global = lv_wall_thick_global.rename(
    columns={'eid': 'IID', '24140-2.0': 'lv_mean_wall_thick_global'}
)[['IID', 'lv_mean_wall_thick_global']]

###########################################################
# 5) Merge Everything Together
#
#    Start by merging Asc/Desc Aorta Distensibility,
#    then merge each LV phenotype DataFrame.
###########################################################
df_IDP = asc_aorta_dist.merge(desc_aorta_dist, on='IID', how='inner')

# Merge LV Structure/Function
df_IDP = df_IDP.merge(lv_ef, on='IID', how='inner')
df_IDP = df_IDP.merge(lv_edv, on='IID', how='inner')
df_IDP = df_IDP.merge(lv_esv, on='IID', how='inner')
df_IDP = df_IDP.merge(lv_sv, on='IID', how='inner')
df_IDP = df_IDP.merge(lv_mass, on='IID', how='inner')

# Merge LV Strain Metrics
df_IDP = df_IDP.merge(lv_circ_strain_global, on='IID', how='inner')
df_IDP = df_IDP.merge(lv_rad_strain_global, on='IID', how='inner')
df_IDP = df_IDP.merge(lv_long_strain_global, on='IID', how='inner')

# Merge LV Wall Thickness
df_IDP = df_IDP.merge(lv_wall_thick_global, on='IID', how='inner')

###########################################################
# 6) Optionally drop any rows with missing values
###########################################################
df_IDP = df_IDP.dropna()

# Inspect the results
print(df_IDP.head())
print(df_IDP.shape)


In [ ]:
# Now `df` contains all relevant columns
df_IDP.head()

In [ ]:
len(df_IDP)

In [ ]:
# Define a function to load and process datasets
def load_and_process(file_path, embedding_col):
    df = pd.read_csv(file_path).rename(columns={'embedding': embedding_col}).drop_duplicates()
    df[embedding_col] = df[embedding_col].apply(lambda x: np.array(json.loads(x)))
    return df



# Process train data for each channel
train_fin = load_and_process('train_embeddings_long_axis_ch0_imbalanced_monai.csv', 'embedding0')
train_fin1 = load_and_process('train_embeddings_long_axis_ch1_imbalanced_monai.csv', 'embedding1')
train_fin2 = load_and_process('train_embeddings_long_axis_ch2_imbalanced_monai.csv', 'embedding2')

# Merge train DataFrames
train_df_x = pd.merge(train_fin, train_fin1, on=['image_path', 'label'], how='inner')
train_df_heart = pd.merge(train_df_x, train_fin2, on=['image_path', 'label'], how='inner')

# Extract IID from image path for train
train_df_heart['IID'] = train_df_heart['image_path'].str.extract(r'Processed/(\d+).npy').astype(int)

# Process test data for each channel
test_fin = load_and_process('test_embeddings_long_axis_ch0_imbalanced_monai.csv', 'embedding0')
test_fin1 = load_and_process('test_embeddings_long_axis_ch1_imbalanced_monai.csv', 'embedding1')
test_fin2 = load_and_process('test_embeddings_long_axis_ch2_imbalanced_monai.csv', 'embedding2')

# Merge test DataFrames
test_df_x = pd.merge(test_fin, test_fin1, on=['image_path', 'label'], how='inner')
test_df_heart = pd.merge(test_df_x, test_fin2, on=['image_path', 'label'], how='inner')

# Extract IID from image path for test
test_df_heart['IID'] = test_df_heart['image_path'].str.extract(r'Processed/(\d+).npy').astype(int)

# Print results (optional, for verification)
print(train_df_heart['label'].value_counts())
print(test_df_heart['label'].value_counts())

In [ ]:
# Define a function to load and process datasets
def load_and_process(file_path, embedding_col):
    df = pd.read_csv(file_path).rename(columns={'embedding': embedding_col}).drop_duplicates()
    df[embedding_col] = df[embedding_col].apply(lambda x: np.array(json.loads(x)))
    return df

# Process train and test datasets
train_df_aortic = load_and_process('train_embeddings_aortic_monai.csv', 'aortic_embeddings')
test_df_aortic = load_and_process('test_embeddings_aortic_monai.csv', 'aortic_embeddings')

# Extract IID from image path for both train and test
train_df_aortic['IID'] = train_df_aortic['image_path'].str.extract(r'/(\d+)_\d+\.npy').astype(int)
test_df_aortic['IID'] = test_df_aortic['image_path'].str.extract(r'/(\d+)_\d+\.npy').astype(int)  # Fixed parentheses

# Print results (optional, for verification)
print(train_df_aortic['label'].value_counts())
print(test_df_aortic['label'].value_counts())

In [ ]:
# List of datasets to merge sequentially
datasets = [
    train_df_heart,
    train_df_aortic
]

# Drop the 'image path' column if it exists in any dataset
datasets = [df.drop(columns=['image_path'], errors='ignore') for df in datasets]


train_df_full = reduce(lambda left, right: pd.merge(left, right, on=['IID', 'label'], how='inner'), datasets)

# Display the resulting dataframe
train_df_full.head()

In [ ]:
# List of test datasets to merge sequentially
test_datasets = [
    test_df_heart,
    test_df_aortic
]

# Drop the 'image path' column if it exists in any dataset
test_datasets = [df.drop(columns=['image_path'], errors='ignore') for df in test_datasets]

test_df_full = reduce(lambda left, right: pd.merge(left, right, on=['IID', 'label'], how='inner'), test_datasets)

# Display the resulting dataframe
test_df_full.head()

In [ ]:
def convert_to_array(embedding):
    """
    Converts a string representation of a list into a NumPy array.
    
    Args:
        embedding (str or list): Embedding to convert.
        
    Returns:
        np.ndarray: Converted embedding.
    """
    if isinstance(embedding, str):
        return np.array(ast.literal_eval(embedding))  # Safely parse string to list
    elif isinstance(embedding, (list, np.ndarray)):
        return np.array(embedding)
    else:
        raise ValueError(f"Unsupported embedding format: {type(embedding)}")

In [ ]:
def fit_pca_on_train(df, embedding_cols, n_components=10):
    """
    Fits PCA on the train embeddings and adds PCA-transformed features to the DataFrame.

    Args:
        df (pd.DataFrame): DataFrame containing the embeddings.
        embedding_cols (list of str): Column names with embeddings.
        n_components (int): Number of PCA components to fit.

    Returns:
        tuple: Updated DataFrame with PCA features and a dictionary of fitted PCA objects.
    """
    pca_dict = {}
    for col in embedding_cols:
        # Ensure the embeddings are numpy arrays
        embeddings = np.vstack(df[col].values)
        
        # Fit PCA
        pca = PCA(n_components=n_components)
        pca.fit(embeddings)
        pca_dict[col] = pca
        
        # Transform and add PCA features to the DataFrame
        pca_features = pca.transform(embeddings)
        pca_feature_cols = [f'pca_{col}_{i+1}' for i in range(n_components)]
        df[pca_feature_cols] = pca_features
    
    return df, pca_dict

In [ ]:
def transform_embeddings_with_pca(df, embedding_cols, pca_dict):
    """
    Transforms embeddings using pre-fitted PCA and adds PCA features to the DataFrame.

    Args:
        df (pd.DataFrame): DataFrame containing the embeddings to transform.
        embedding_cols (list of str): Column names with embeddings.
        pca_dict (dict): Dictionary of pre-fitted PCA objects for each column.

    Returns:
        pd.DataFrame: Updated DataFrame with PCA-transformed features added.
    """
    for col in embedding_cols:
        # Ensure the embeddings are numpy arrays
        embeddings = np.vstack(df[col].values)
        
        # Transform using the pre-fitted PCA
        pca = pca_dict[col]
        pca_features = pca.transform(embeddings)
        
        # Add PCA features to the DataFrame
        pca_feature_cols = [f'pca_{col}_{i+1}' for i in range(pca.n_components)]
        df[pca_feature_cols] = pca_features
    
    return df

In [ ]:
# Embedding Columns and PCA
embedding_columns = [
    'embedding0', 
    'embedding1', 
    'embedding2', 
    'aortic_embeddings'
    
]
# Convert all embeddings to NumPy arrays
for col in embedding_columns:
    train_df_full[col] = train_df_full[col].apply(convert_to_array)


emb_columns  = [col for col in train_df_full.columns if 'embedding' in col.lower()]

# Perform PCA on the embedding columns
train_df_full, pca_dict = fit_pca_on_train(train_df_full, embedding_columns)


# Generate PCA columns dynamically
pca_columns = [f'pca_{col}_{i+1}' for col in embedding_columns for i in range(10)]



# Select relevant columns for analysis
columns_to_keep = [
    'IID', 'label'
] + pca_columns +emb_columns
train_df = train_df_full[columns_to_keep]

In [ ]:
train_df.head()

In [ ]:
idp_df = pd.merge(train_df, df_IDP, on= 'IID', how = 'inner')

In [ ]:
len(idp_df)

In [ ]:
idp_df['label'].value_counts()

In [ ]:
idp_df.head()

In [ ]:
idp_df.columns

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

##############################################################################
# 1) Flatten Raw Embeddings into Separate Columns
##############################################################################
def expand_embedding_columns(df, col_name):
    """
    Expands a column containing 512-dimensional array embeddings into
    512 separate columns.
    """
    emb_matrix = np.vstack(df[col_name].apply(lambda x: np.array(x) if isinstance(x, list) else np.zeros(512)))
    emb_df = pd.DataFrame(emb_matrix, columns=[f"{col_name}_{i+1}" for i in range(512)])
    return emb_df

# Expand all 4 embedding vectors (512 each)
embedding0_df = expand_embedding_columns(idp_df, 'embedding0')
embedding1_df = expand_embedding_columns(idp_df, 'embedding1')
embedding2_df = expand_embedding_columns(idp_df, 'embedding2')
aortic_embeddings_df = expand_embedding_columns(idp_df, 'aortic_embeddings')

# Merge back into idp_df
idp_df = pd.concat([idp_df, embedding0_df, embedding1_df, embedding2_df, aortic_embeddings_df], axis=1)

##############################################################################
# 2) Identify Feature Groups
##############################################################################
idp_cols = [
    'asc_aorta_dist', 'desc_aorta_dist',
    'lv_ejection_fraction', 'lv_end_diastolic_vol', 'lv_end_systolic_vol',
    'lv_stroke_vol', 'lv_myocardial_mass',
    'lv_circ_strain_global', 'lv_rad_strain_global', 'lv_long_strain_global',
    'lv_mean_wall_thick_global'
]

# PCA Embedding columns
pca_embedding_cols = [col for col in idp_df.columns if 'pca_embedding' in col.lower()]

# Raw Embedding columns
raw_embedding_cols = embedding0_df.columns.tolist() + embedding1_df.columns.tolist() + embedding2_df.columns.tolist() + aortic_embeddings_df.columns.tolist()

label_col = 'label'  

##############################################################################
# 3) 5-Fold CV Function
##############################################################################
def cross_val_auc(X, y, n_splits=5):
    """
    Performs 5-fold cross-validation with XGBoost and returns AUC scores.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_list, accuracy_list, roc_curves = [], [], {}

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Train XGBoost model
        model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

        # Compute metrics
        auc = roc_auc_score(y_test, y_pred_proba)
        accuracy = accuracy_score(y_test, y_pred)
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

        auc_list.append(auc)
        accuracy_list.append(accuracy)
        roc_curves[fold] = (fpr, tpr, auc)

    return np.array(auc_list), np.array(accuracy_list), roc_curves

##############################################################################
# 4) Train Models and Evaluate
##############################################################################

df_model = idp_df[[label_col] + idp_cols + pca_embedding_cols + raw_embedding_cols].dropna()
df_model['outcome'] = df_model[label_col].astype(int)

# Prepare data for each model
X_A = df_model[idp_cols].values
X_B = df_model[raw_embedding_cols].values
X_C = df_model[pca_embedding_cols].values
X_A_B = df_model[idp_cols + raw_embedding_cols].values
X_A_C = df_model[idp_cols + pca_embedding_cols].values
y = df_model['outcome'].values

# Perform 5-fold CV for each model
auc_A, acc_A, roc_A = cross_val_auc(X_A, y)
auc_B, acc_B, roc_B = cross_val_auc(X_B, y)
auc_C, acc_C, roc_C = cross_val_auc(X_C, y)
auc_A_B, acc_A_B, roc_A_B = cross_val_auc(X_A_B, y)
auc_A_C, acc_A_C, roc_A_C = cross_val_auc(X_A_C, y)

##############################################################################
# 5) Print Results
##############################################################################

print("\n=== Cross-Validation AUC and Accuracy ===")
print(f"Model A (IDPs)             : AUC = {auc_A.mean():.3f} ± {auc_A.std():.3f}, Accuracy = {acc_A.mean():.3f} ± {acc_A.std():.3f}")
print(f"Model B (Raw Embeddings)   : AUC = {auc_B.mean():.3f} ± {auc_B.std():.3f}, Accuracy = {acc_B.mean():.3f} ± {acc_B.std():.3f}")
print(f"Model C (PCA Embeddings)   : AUC = {auc_C.mean():.3f} ± {auc_C.std():.3f}, Accuracy = {acc_C.mean():.3f} ± {acc_C.std():.3f}")
print(f"Model A+B (IDPs + Raw Embs): AUC = {auc_A_B.mean():.3f} ± {auc_A_B.std():.3f}, Accuracy = {acc_A_B.mean():.3f} ± {acc_A_B.std():.3f}")
print(f"Model A+C (IDPs + PCA)     : AUC = {auc_A_C.mean():.3f} ± {auc_A_C.std():.3f}, Accuracy = {acc_A_C.mean():.3f} ± {acc_A_C.std():.3f}")

##############################################################################
# 6) Plot ROC Curves
##############################################################################
plt.figure(figsize=(10, 8))

def plot_roc_curve(roc_dict, label):
    mean_fpr = np.linspace(0, 1, 100)
    mean_tpr = np.mean([np.interp(mean_fpr, roc_dict[i][0], roc_dict[i][1]) for i in range(5)], axis=0)
    mean_auc = np.mean([roc_dict[i][2] for i in range(5)])
    plt.plot(mean_fpr, mean_tpr, label=f"{label} (AUC = {mean_auc:.2f})")

plot_roc_curve(roc_A, "IDPs")
plot_roc_curve(roc_C, "PCA Embeddings")
plot_roc_curve(roc_A_C, "IDPs + PCA Embeddings")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.title("ROC Curves for Different Feature Sets (5-Fold CV)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import chi2

############################
# Helper: Manual LRT
############################
def likelihood_ratio_test(model_small, model_large):
    ll_small = model_small.llf
    ll_large = model_large.llf
    df_small = model_small.df_model
    df_large = model_large.df_model

    lr_stat = 2.0 * (ll_large - ll_small)
    df_diff = int(df_large - df_small)
    p_value = chi2.sf(lr_stat, df_diff)
    
    return lr_stat, p_value, df_diff

############################
# 1) Prepare data
############################
df_model = idp_df.copy()
label_col = 'label'
df_model = df_model[[label_col] + idp_cols + pca_embedding_cols].dropna()
df_model = df_model.rename(columns={label_col: 'outcome'})
df_model['outcome'] = df_model['outcome'].astype(int)

# Build formulas
formula_idps = "outcome ~ " + " + ".join(idp_cols)       # Model A
formula_embs = "outcome ~ " + " + ".join(pca_embedding_cols)  # Model B
formula_both = "outcome ~ " + " + ".join(idp_cols + pca_embedding_cols)  # Model C

############################
# 2) Fit logistic models
############################
# Model A: IDPs
model_A = smf.logit(formula_idps, data=df_model).fit(disp=False)
print("=== Model A (IDPs) ===")
print(model_A.summary())

# Model B: Embeddings
model_B = smf.logit(formula_embs, data=df_model).fit(disp=False)
print("\n=== Model B (Embeddings) ===")
print(model_B.summary())

# Model C: IDPs + Embeddings
model_C = smf.logit(formula_both, data=df_model).fit(disp=False)
print("\n=== Model C (IDPs + Embeddings) ===")
print(model_C.summary())

############################
# 3) Likelihood Ratio Tests
############################
lr_stat_A_C, p_val_A_C, df_diff_A_C = likelihood_ratio_test(model_A, model_C)
lr_stat_B_C, p_val_B_C, df_diff_B_C = likelihood_ratio_test(model_B, model_C)

print("\n=== Likelihood Ratio Tests (Manual) ===")
print(f"Model A -> C: LR-stat={lr_stat_A_C:.3f}, df={df_diff_A_C}, p={p_val_A_C:.6f}")
print(f"Model B -> C: LR-stat={lr_stat_B_C:.3f}, df={df_diff_B_C}, p={p_val_B_C:.6f}")

############################
# 4) Cross-Validation AUC
############################
def cross_val_auc(X, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    clf = LogisticRegression(max_iter=1000, solver='liblinear')
    auc_list = []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        clf.fit(X_train, y_train)
        y_proba = clf.predict_proba(X_test)[:,1]
        auc_list.append(roc_auc_score(y_test, y_proba))
    return np.array(auc_list)

X_A = df_model[idp_cols].values
X_B = df_model[pca_embedding_cols].values
X_C = df_model[idp_cols + pca_embedding_cols].values
y   = df_model['outcome'].values

auc_scores_A = cross_val_auc(X_A, y)
auc_scores_B = cross_val_auc(X_B, y)
auc_scores_C = cross_val_auc(X_C, y)

print("\n=== Cross-Validation AUC ===")
print(f"Model A (IDPs)       : {auc_scores_A.mean():.3f} ± {auc_scores_A.std():.3f}")
print(f"Model B (PCs) : {auc_scores_B.mean():.3f} ± {auc_scores_B.std():.3f}")
print(f"Model C (Combined)   : {auc_scores_C.mean():.3f} ± {auc_scores_C.std():.3f}")

############################
# 5) Visualization
############################
results_df = pd.DataFrame({
    'AUC': np.concatenate([auc_scores_A, auc_scores_B, auc_scores_C]),
    'Model': (['A (IDPs)'] * len(auc_scores_A)
             + ['B (PCs)'] * len(auc_scores_B)
             + ['C (Combined)'] * len(auc_scores_C))
})

plt.figure(figsize=(7,6))
sns.boxplot(data=results_df, x='Model', y='AUC', palette='Set2')
sns.stripplot(data=results_df, x='Model', y='AUC', color='black', alpha=0.6)
plt.title('Cross-Validated AUC Comparison')
plt.ylabel('AUC')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

############################
# Helper: Likelihood Ratio Test Function
############################
def likelihood_ratio_test(model_small, model_large):
    ll_small = model_small.llf
    ll_large = model_large.llf
    df_small = model_small.df_model
    df_large = model_large.df_model

    lr_stat = 2.0 * (ll_large - ll_small)
    df_diff = int(df_large - df_small)
    p_value = chi2.sf(lr_stat, df_diff)
    
    return lr_stat, p_value, df_diff

############################
# 1) Prepare Data
############################
df_model = idp_df.copy()
label_col = 'label'
df_model = df_model[[label_col] + idp_cols + pca_embedding_cols].dropna()
df_model = df_model.rename(columns={label_col: 'outcome'})
df_model['outcome'] = df_model['outcome'].astype(int)

# Build formulas
formula_A = "outcome ~ " + " + ".join(idp_cols)               # Model A (IDPs)
formula_B = "outcome ~ " + " + ".join(pca_embedding_cols)     # Model B (PCA)
formula_C = "outcome ~ " + " + ".join(idp_cols + pca_embedding_cols)  # Model C (IDPs + PCA)

############################
# 2) Fit Logistic Regression Models
############################
# Model A: IDPs only
model_A = smf.logit(formula_A, data=df_model).fit(disp=False)

# Model B: PCA Embeddings only
model_B = smf.logit(formula_B, data=df_model).fit(disp=False)

# Model C: IDPs + PCA Embeddings
model_C = smf.logit(formula_C, data=df_model).fit(disp=False)

############################
# 3) Likelihood Ratio Tests (A→B, A→C, B→C)
############################
lr_stat_A_B, p_val_A_B, df_diff_A_B = likelihood_ratio_test(model_A, model_B)
lr_stat_A_C, p_val_A_C, df_diff_A_C = likelihood_ratio_test(model_A, model_C)
lr_stat_B_C, p_val_B_C, df_diff_B_C = likelihood_ratio_test(model_B, model_C)

lrt_results = pd.DataFrame({
    "Comparison": ["A → B", "A → C", "B → C"],
    "LR-Stat": [lr_stat_A_B, lr_stat_A_C, lr_stat_B_C],
    "DF Diff": [df_diff_A_B, df_diff_A_C, df_diff_B_C],
    "P-Value": [p_val_A_B, p_val_A_C, p_val_B_C]
})

# Save extended LRT results
lrt_results.to_csv("likelihood_ratio_tests_extended.csv", index=False)

############################
# 4) Visualization: LRT Results Bar Plot
############################
plt.figure(figsize=(8,6))
sns.barplot(data=lrt_results, x="Comparison", y="LR-Stat", palette="coolwarm", alpha=0.8)

# Annotate p-values on top of the bars
for i, p in enumerate(lrt_results["P-Value"]):
    plt.text(i, lrt_results["LR-Stat"][i] + 1, f"p = {p:.3e}", ha='center', fontsize=12)

plt.title("Likelihood Ratio Test Results")
plt.ylabel("Likelihood Ratio Statistic")
plt.xlabel("Model Comparison")
plt.grid(axis='y', linestyle="--", alpha=0.6)

# Save the updated LRT results plot
plt.savefig("lrt_results_extended_plot.png", dpi=300, bbox_inches="tight")
plt.show()
